# Daum Movie Review Notebook

`daum_movie_review.csv`를 읽어서 감정분류용 데이터로 바꾸고, `preprocessing.py`와 `model.py`를 연결해 실행하는 노트북이다.

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from preprocessing import DEFAULT_STOPWORDS, KoreanSentimentDataset, preprocess_dataframe
from model import BiLSTMSentimentClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

c:\Users\Playdata\AppData\Local\miniconda3\envs\new_01\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cpu')

In [2]:
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [3]:
print(df.columns)

review_col = 'review'
rating_col = 'rating'

data = df[[review_col, rating_col]].copy()
data.columns = ['review', 'rating']
data['sentiment'] = (data['rating'] >= 6).astype(int)
data[['review', 'rating', 'sentiment']].head()

Index(['review', 'rating', 'date', 'title'], dtype='object')


,review,rating,sentiment
0,돈 들인건 티가 나지만 보는 내내 하품만,1,0
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,1
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,1
3,이 정도면 볼만하다고 할 수 있음!,8,1
4,재미있다,10,1


In [4]:
processed_df, vocab = preprocess_dataframe(
    df=data[['review', 'sentiment']],
    text_col='review',
    label_col='sentiment',
    max_len=30,
    stopwords=DEFAULT_STOPWORDS,
    min_freq=1,
)

dataset = KoreanSentimentDataset(processed_df)
print('vocab size:', len(vocab))
print('dataset size:', len(dataset))
print(processed_df[['review', 'clean_text', 'tokens', 'padded_ids', 'length', 'label']].head())

vocab size: 54967
dataset size: 14558
                                              review  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1       몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...   
3                                이 정도면 볼만하다고 할 수 있음!   
4                                               재미있다   

                                          clean_text  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1          몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...   
3                                 이 정도면 볼만하다고 할 수 있음   
4                                               재미있다   

                                              tokens  \
0                     [돈, 들인건, 티가, 나지만, 보는, 내내, 하품만]   
1  [몰입할수밖에, 없다, 어렵게, 생각할, 필요없다, 내가, 전투에, 참여한듯, 손에...   
2  [이전, 작품에, 비해, 화려하고, 스케일도, 커졌지만, 전국, 맛집의, 음식들을,...   
3                                [정도면, 볼만하다고, 할, 있음]   
4       

In [5]:
val_size = max(1, int(len(dataset) * 0.2)) if len(dataset) > 1 else 0
train_size = len(dataset) - val_size

if val_size > 0:
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
else:
    train_dataset = dataset
    val_dataset = dataset

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = BiLSTMSentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
    pad_idx=vocab['<pad>'],
    verbose=False,
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

BiLSTMSentimentClassifier(
  (embedding): Embedding(54967, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [6]:
def accuracy_from_probs(probs: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

model.train()
for batch_idx, (input_ids, labels, lengths) in enumerate(train_loader):
    input_ids = input_ids.to(device)
    labels = labels.to(device).float().unsqueeze(1)
    lengths = lengths.to(device)

    optimizer.zero_grad(set_to_none=True)
    probs = model(input_ids, lengths)
    loss = criterion(probs, labels)
    loss.backward()
    optimizer.step()

    print('train loss:', float(loss.item()))
    print('train acc :', accuracy_from_probs(probs.detach(), labels.detach()))    

model.eval()
with torch.no_grad():
    for batch_idx, (input_ids, labels, lengths) in enumerate(val_loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        lengths = lengths.to(device)
        probs = model(input_ids, lengths)
        print('val acc:', accuracy_from_probs(probs, labels))        

train loss: 0.720641016960144
train acc : 0.21875
train loss: 0.6978724598884583
train acc : 0.5
train loss: 0.7038443088531494
train acc : 0.375
train loss: 0.6917107105255127
train acc : 0.46875
train loss: 0.6877875328063965
train acc : 0.5625
train loss: 0.6873346567153931
train acc : 0.5625
train loss: 0.6699970960617065
train acc : 0.625
train loss: 0.6875628232955933
train acc : 0.53125
train loss: 0.6854224801063538
train acc : 0.59375
train loss: 0.6678946614265442
train acc : 0.59375
train loss: 0.6699685454368591
train acc : 0.71875
train loss: 0.6466879844665527
train acc : 0.78125
train loss: 0.6730166673660278
train acc : 0.59375
train loss: 0.6277766227722168
train acc : 0.78125
train loss: 0.6827019453048706
train acc : 0.59375
train loss: 0.657357394695282
train acc : 0.71875
train loss: 0.6633105874061584
train acc : 0.65625
train loss: 0.6427488923072815
train acc : 0.6875
train loss: 0.6656892895698547
train acc : 0.65625
train loss: 0.5949356555938721
train acc : 0

In [7]:
# epochs 정의해서 원하는 epoch만큼 학습과 val 출력
# 최종 test 성능
# 원하는 문장 넣어 추론